# IPrakriti — Exploratory Data Analysis (EDA)

Run this notebook **after** your teammates finish collecting data and you export the Google Form CSV.

This notebook helps you:
1. Understand the distribution of Prakriti labels
2. Spot data quality issues before training
3. See which questions are most informative

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data.preprocess import (
    QUESTION_COLUMNS, TARGET_COLUMN, LABEL_DECODING,
    generate_dummy_data, run_pipeline
)

# ---------------------------------------------------------------
# CHANGE THIS: set USE_DUMMY = False and update CSV_PATH
# once your real data is in data/raw/
# ---------------------------------------------------------------
USE_DUMMY = True
CSV_PATH  = '../data/raw/prakriti_responses.csv'

if USE_DUMMY:
    df_raw = generate_dummy_data(n_samples=300)
    print('Using SYNTHETIC data (300 samples)')
else:
    df_raw = pd.read_csv(CSV_PATH)
    print(f'Loaded real data: {df_raw.shape}')

df_raw.head()

## 1. Label Distribution
Are all Prakriti types well represented?

In [ ]:
label_counts = df_raw[TARGET_COLUMN].map(LABEL_DECODING).value_counts()
fig, ax = plt.subplots(figsize=(9, 4))
label_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Prakriti Label Distribution')
ax.set_xlabel('Prakriti Type')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=30)
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            int(bar.get_height()), ha='center', fontsize=9)
plt.tight_layout()
plt.show()
print(label_counts.to_string())
print(f'\nMinority/Majority ratio: {label_counts.min()/label_counts.max():.2f}')
print('(Below 0.5 means significant imbalance — SMOTE will help)')

## 2. Missing Values
Any questions that respondents skipped?

In [ ]:
nulls = df_raw[QUESTION_COLUMNS].isna().sum()
if nulls.sum() == 0:
    print('✅ No missing values found')
else:
    print('⚠️ Questions with missing answers:')
    print(nulls[nulls > 0].to_string())

## 3. Answer Distribution Per Question
Check if any question has very unbalanced answer choices

In [ ]:
fig, axes = plt.subplots(7, 5, figsize=(18, 20))
axes = axes.flatten()
colors = ['#5B8DB8', '#E07B54', '#6AAB6E']
for i, col in enumerate(QUESTION_COLUMNS):
    counts = df_raw[col].value_counts().sort_index()
    axes[i].bar(counts.index, counts.values,
                color=[colors[j] for j in counts.index], edgecolor='white')
    axes[i].set_title(col.replace('_', ' '), fontsize=7)
    axes[i].set_xticks([0, 1, 2])
    axes[i].set_xticklabels(['V', 'P', 'K'], fontsize=8)
plt.suptitle('Answer Distribution (V=Vata, P=Pitta, K=Kapha)', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 4. Correlation Heatmap
Which questions tend to be answered similarly?

In [ ]:
corr = df_raw[QUESTION_COLUMNS].corr()
fig, ax = plt.subplots(figsize=(15, 12))
sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0,
            xticklabels=[c.split('_')[0] for c in QUESTION_COLUMNS],
            yticklabels=[c.split('_')[0] for c in QUESTION_COLUMNS],
            linewidths=0.3)
ax.set_title('Question Correlation Heatmap')
plt.tight_layout()
plt.show()

## 5. Quick Sanity Check — Does the data make sense?
Vata answers should cluster at 0, Pitta at 1, Kapha at 2

In [ ]:
pure_doshas = df_raw[df_raw[TARGET_COLUMN].isin([0, 1, 2])].copy()
means = pure_doshas.groupby(TARGET_COLUMN)[QUESTION_COLUMNS].mean()
means.index = means.index.map(LABEL_DECODING)
fig, ax = plt.subplots(figsize=(15, 4))
means.T.plot(ax=ax, marker='o', linewidth=1.5)
ax.set_title('Mean Answer per Dosha (should be: Vata≈0, Pitta≈1, Kapha≈2)')
ax.set_xlabel('Question')
ax.set_ylabel('Mean Answer')
ax.set_xticks(range(35))
ax.set_xticklabels([f'Q{i+1}' for i in range(35)], rotation=45, fontsize=7)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()
print('If lines are well separated → data quality is good ✅')
print('If lines overlap heavily  → review question encoding in preprocess.py')